# cjm-transcription-source-select

> FastHTML source selection component for transcription workflows that provides local file browsing with audio/video preview, selection ordering, and integrated audio extraction from video sources.

## Install

```bash
pip install cjm_transcription_source_select
```

## Project Structure

```
nbs/
├── components/ (4)
│   ├── file_browser_panel.ipynb  # File browser panel configuration and rendering for browsing local audio/video files
│   ├── helpers.ipynb             # Shared helper functions for the transcription source selection step
│   ├── preview_panel.ipynb       # Collapsible preview panel with audio/video player and file metadata
│   └── selection_panel.ipynb     # Selection panel component showing selected files with drag-drop reordering
├── routes/ (4)
│   ├── browser.ipynb    # Route handlers for file browser navigation and file selection
│   ├── core.ipynb       # State management helpers for the transcription source selection step
│   ├── preview.ipynb    # Route handlers for media file serving and preview panel rendering
│   └── selection.ipynb  # Route handlers for selection queue management (remove, reorder, clear)
├── html_ids.ipynb  # HTML ID constants for the transcription source selection step
├── models.ipynb    # Data models and URL bundles for the transcription source selection step
└── utils.ipynb     # Utility functions for file type detection, duration formatting, and extension filtering
```

Total: 11 notebooks across 3 directories

## Module Dependencies

```mermaid
graph LR
    components_file_browser_panel[components.file_browser_panel<br/>components/file_browser_panel]
    components_helpers[components.helpers<br/>components/helpers]
    components_preview_panel[components.preview_panel<br/>components/preview_panel]
    components_selection_panel[components.selection_panel<br/>components/selection_panel]
    html_ids[html_ids<br/>html_ids]
    models[models<br/>models]
    routes_browser[routes.browser<br/>routes/browser]
    routes_core[routes.core<br/>routes/core]
    routes_preview[routes.preview<br/>routes/preview]
    routes_selection[routes.selection<br/>routes/selection]
    utils[utils<br/>utils]

    components_file_browser_panel --> html_ids
    components_preview_panel --> html_ids
    components_preview_panel --> utils
    components_preview_panel --> models
    components_selection_panel --> models
    components_selection_panel --> html_ids
    components_selection_panel --> utils
    routes_browser --> models
    routes_browser --> utils
    routes_browser --> components_file_browser_panel
    routes_browser --> routes_core
    routes_browser --> components_selection_panel
    routes_core --> models
    routes_preview --> models
    routes_preview --> routes_core
    routes_preview --> components_preview_panel
    routes_selection --> components_file_browser_panel
    routes_selection --> models
    routes_selection --> routes_core
    routes_selection --> components_selection_panel
```

*20 cross-module dependencies detected*

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### routes/browser (`browser.ipynb`)
> Route handlers for file browser navigation and file selection

#### Import

```python
from cjm_transcription_source_select.routes.browser import (
    init_browser_router
)
```

#### Functions

```python
def _handle_navigate(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # Directory path to navigate to
):  # Rendered browser panel
    "Navigate to a directory and re-render the browser panel."
```

```python
def _handle_select(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to toggle
):  # (browser panel, OOB selection panel)
    "Toggle a file in/out of the selected files list."
```

```python
def init_browser_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle (populated after router init)
    home_path: str = "",  # Home directory for nav buttons
    prefix: str = "/browser",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize the file browser router with navigate and select handlers."
```


### routes/core (`core.ipynb`)
> State management helpers for the transcription source selection step

#### Import

```python
from cjm_transcription_source_select.routes.core import (
    STEP_KEY,
    get_step_state,
    update_step_state,
    get_session_id_from_sess
)
```

#### Functions

```python
def get_step_state(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    session_id: str,  # Session identifier
) -> SourceSelectState:  # Current step state (empty dict if not found)
    "Retrieve the source selection step state."
```

```python
def update_step_state(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    session_id: str,  # Session identifier
    **fields,  # Fields to update in the step state
) -> SourceSelectState:  # Updated step state
    "Update specific fields in the source selection step state."
```

```python
def get_session_id_from_sess(
    sess: Any  # Session object from FastHTML
) -> str:  # Session identifier string
    "Extract session ID from a FastHTML session object."
```

#### Variables

```python
STEP_KEY = 'source_select'
```

### components/file_browser_panel (`file_browser_panel.ipynb`)
> File browser panel configuration and rendering for browsing local audio/video files

#### Import

```python
from cjm_transcription_source_select.components.file_browser_panel import (
    AUDIO_FILTER_EXTENSIONS,
    VIDEO_FILTER_EXTENSIONS,
    MEDIA_FILTER_EXTENSIONS,
    create_media_browser_config,
    get_browser_state,
    sync_browser_selection,
    render_browser_panel
)
```

#### Functions

```python
def create_media_browser_config() -> FileBrowserConfig:  # Configured for audio/video file selection
    "Create file browser config for audio/video file selection."
```

```python
def get_browser_state(
    step_state: Dict[str, Any],  # Current step state
    default_path: str = "",  # Default directory if no state exists
) -> BrowserState:  # Browser state for the file browser
    "Get or create BrowserState from step state."
```

```python
def sync_browser_selection(
    browser_state: BrowserState,  # Browser state to update
    selected_files: List[Dict[str, Any]],  # Current selected_files from step state
) -> None:  # Modifies browser_state in place
    "Sync browser selection state with the selected_files list."
```

```python
def render_browser_panel(
    browser_state: BrowserState,  # Current browser state
    config: FileBrowserConfig,  # Browser configuration
    provider: LocalFileSystemProvider,  # File system provider
    navigate_url: str,  # URL for directory navigation
    select_url: str,  # URL for file selection toggle
    home_path: str = "",  # Home directory for nav buttons
) -> Any:  # Rendered file browser component
    "Render the file browser panel using the library's built-in UI."
```

#### Variables

```python
AUDIO_FILTER_EXTENSIONS = [8 items]
VIDEO_FILTER_EXTENSIONS = [7 items]
MEDIA_FILTER_EXTENSIONS
```

### components/helpers (`helpers.ipynb`)
> Shared helper functions for the transcription source selection step

#### Import

```python
from cjm_transcription_source_select.components.helpers import (
    generate_sortable_init_script
)
```

#### Functions

```python
def generate_sortable_init_script(
    container_selector: str = ".sortable",  # CSS selector for sortable containers
    handle_selector: str = ".drag-handle",  # CSS selector for drag handles
    animation_ms: int = 150,  # Animation duration in milliseconds
) -> str:  # JavaScript initialization script
    "Generate Sortable.js initialization script for htmx integration."
```


### html_ids (`html_ids.ipynb`)
> HTML ID constants for the transcription source selection step

#### Import

```python
from cjm_transcription_source_select.html_ids import (
    SourceSelectHtmlIds
)
```
#### Classes

```python
class SourceSelectHtmlIds:
    "HTML ID constants for the transcription source selection step."
    
    def as_selector(
            id_str: str  # The HTML ID to convert
        ) -> str:  # CSS selector with # prefix
        "Convert an ID to a CSS selector format."
    
    def file_item(
            file_path: str  # Absolute path to the file
        ) -> str:  # HTML ID for the file item in the browser
        "Generate HTML ID for a file item in the browser list."
    
    def selection_item(
            file_path: str  # Absolute path to the file
        ) -> str:  # HTML ID for the selection list item
        "Generate HTML ID for a selected file item."
    
    def extraction_status(
            file_path: str  # Absolute path to the video file
        ) -> str:  # HTML ID for the extraction status indicator
        "Generate HTML ID for a video file's extraction status."
```


### models (`models.ipynb`)
> Data models and URL bundles for the transcription source selection step

#### Import

```python
from cjm_transcription_source_select.models import (
    SelectedFile,
    ExtractionResult,
    SourceSelectState,
    SourceSelectUrls
)
```
#### Classes

```python
class SelectedFile(TypedDict):
    "A selected audio or video file."
```

```python
class ExtractionResult(TypedDict):
    "Audio extraction result for a video file."
```

```python
class SourceSelectState(TypedDict):
    "State for the transcription source selection step."
```

```python
@dataclass
class SourceSelectUrls:
    "URL bundle for transcription source selection routes."
    
    navigate: str = ''  # Navigate to directory
    select: str = ''  # Toggle file selection (from file browser)
    remove: str = ''  # Remove file from selection
    reorder: str = ''  # Reorder selection (SortableJS)
    clear: str = ''  # Clear all selected files
    preview: str = ''  # Preview a file (render player)
    media_src: str = ''  # Serve a local media file for HTML5 players
    verify: str = ''  # Verify selection + trigger extraction
    extraction_status: str = ''  # Poll extraction status
```


### routes/preview (`preview.ipynb`)
> Route handlers for media file serving and preview panel rendering

#### Import

```python
from cjm_transcription_source_select.routes.preview import (
    init_preview_router
)
```

#### Functions

```python
def _handle_media_src(
    path: str,  # Absolute path to the media file
) -> Response:  # FileResponse or 404
    "Serve a local media file with appropriate MIME type."
```

```python
def _handle_preview(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to preview
):  # Rendered preview panel
    "Render the preview panel for a selected file."
```

```python
def init_preview_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    prefix: str = "/preview",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize preview routes for media serving and preview panel rendering."
```


### components/preview_panel (`preview_panel.ipynb`)
> Collapsible preview panel with audio/video player and file metadata

#### Import

```python
from cjm_transcription_source_select.components.preview_panel import (
    render_preview_panel
)
```

#### Functions

```python
def _render_metadata_row(
    label: str,  # Label text
    value: str,  # Value text
) -> Div:  # Metadata row element
    "Render a single metadata label-value pair."
```

```python
def _render_file_metadata(
    selected_file: SelectedFile,  # File to display metadata for
) -> Div:  # Metadata section
    "Render file metadata section."
```

```python
def render_preview_panel(
    selected_file: Optional[SelectedFile] = None,  # File to preview (None for empty state)
    media_src_url: str = "",  # Base URL for media file serving
    is_open: bool = False,  # Whether panel starts open
) -> Div:  # Preview panel component
    "Render a collapsible preview panel with audio/video player."
```


### routes/selection (`selection.ipynb`)
> Route handlers for selection queue management (remove, reorder, clear)

#### Import

```python
from cjm_transcription_source_select.routes.selection import (
    init_selection_router
)
```

#### Functions

```python
def _render_oob_browser(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    session_id: str,  # Session identifier
    selected_files: list,  # Current selected files
):  # Browser panel with OOB attribute set
    "Render browser panel as OOB swap to update selection highlights."
```

```python
def _handle_remove(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to remove
):  # (selection panel, OOB browser panel)
    "Remove a file from the selection."
```

```python
async def _handle_reorder(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    request,  # FastHTML request
    sess,  # FastHTML session
):  # Rendered selection panel
    "Reorder selected files based on SortableJS drag result."
```

```python
def _handle_clear(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
):  # (selection panel, OOB browser panel)
    "Clear all selected files."
```

```python
def init_selection_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider (for OOB browser updates)
    config: FileBrowserConfig,  # Browser configuration (for OOB browser updates)
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    home_path: str = "",  # Home directory path
    prefix: str = "/selection",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize selection queue management routes."
```


### components/selection_panel (`selection_panel.ipynb`)
> Selection panel component showing selected files with drag-drop reordering

#### Import

```python
from cjm_transcription_source_select.components.selection_panel import (
    render_queue_item,
    render_selection_panel
)
```

#### Functions

```python
def _render_type_badge(
    file_type: str,  # "audio" or "video"
) -> Span:  # Badge component
    "Render a type badge for a selected file."
```

```python
def render_queue_item(
    selected_file: SelectedFile,  # Selected file data
    index: int,  # Position in queue (1-based)
    urls: SourceSelectUrls,  # URL bundle
) -> Li:  # Queue item element
    "Render a single item in the selection queue."
```

```python
def render_selection_panel(
    selected_files: List[SelectedFile],  # Ordered list of selected files
    urls: SourceSelectUrls,  # URL bundle
) -> Div:  # Selection panel component
    "Render the selection panel with drag-drop reordering."
```


### utils (`utils.ipynb`)
> Utility functions for file type detection, duration formatting, and extension filtering

#### Import

```python
from cjm_transcription_source_select.utils import (
    AUDIO_EXTENSIONS,
    VIDEO_EXTENSIONS,
    MEDIA_EXTENSIONS,
    detect_file_type,
    is_media_file,
    format_duration
)
```

#### Functions

```python
def detect_file_type(
    file_path: str  # Path to the file
) -> Optional[str]:  # "audio", "video", or None if not a media file
    "Detect whether a file is audio, video, or not a media file based on extension."
```

```python
def is_media_file(
    file_path: str  # Path to the file
) -> bool:  # True if the file has a supported media extension
    "Check if a file has a supported audio or video extension."
```

```python
def format_duration(
    seconds: Optional[float]  # Duration in seconds, or None
) -> str:  # Formatted duration string (e.g. "1:23:45" or "42:17")
    "Format a duration in seconds as H:MM:SS or MM:SS."
```

#### Variables

```python
AUDIO_EXTENSIONS
VIDEO_EXTENSIONS
MEDIA_EXTENSIONS
```